# ShotGuide 최종 인스타그램 파이프라인

이 노트북은 최종 적용 모델인 `CLIP stronger head`를 사용해 인스타그램 링크에서 overlay 영상까지 생성합니다.

전체 흐름:

```text
Instagram link
→ mp4 다운로드
→ CLIP distance 기반 scene detection
→ scene별 대표 frame 추출
→ shot_type / has_text 예측
→ 촬영 가이드 문구 생성
→ overlay mp4 생성
```




In [1]:
from pathlib import Path
import pandas as pd
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import shotguide_final_instagram_pipeline as pipeline
print("ROOT:", ROOT)
print("최종 모델 checkpoint:", ROOT / "checkpoints" / "clip_vit_b32_multitask_head.pt")




c:\Users\eunpa\anaconda3\envs\sy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT: c:\Temp\shot-classification
최종 모델 checkpoint: c:\Temp\shot-classification\outputs_clip_embeddings\clip_vit_b32_multitask_head.pt


## 1. 분석할 인스타그램 링크 입력

아래 리스트에 새 링크를 추가한 뒤 다음 셀을 실행합니다.




In [2]:
INSTAGRAM_LINKS = [
    "https://www.instagram.com/reels/DYRs6scPeVC/",
]

# scene 하나에서 대표 frame을 몇 장 사용할지 설정합니다.
pipeline.NUM_FRAME_SAMPLES = 3

# 다운로드 파일 번호 시작값입니다.
pipeline.START_INDEX = 1

INSTAGRAM_LINKS




['https://www.instagram.com/reels/DYRs6scPeVC/']

## 2. 전체 파이프라인 실행

실행 결과는 `outputs/final_instagram_pipeline/` 아래에 저장됩니다.

생성 결과:

```text
outputs/final_instagram_pipeline/
  new_001_SHORTCODE/
    scene_frames/
    clip_distance_profile.csv
    scene_metadata.csv
    scene_predictions.csv
    scene_clip_embeddings.npz
    new_001_SHORTCODE_overlay.mp4
  final_pipeline_summary.csv
  final_pipeline_failed_links.csv  # 실패 링크가 있을 때만 생성
```




In [3]:
summary_df, failed_df = pipeline.run_pipeline(INSTAGRAM_LINKS)

print("성공:", len(summary_df))
print("실패:", len(failed_df))

summary_df




c:\Users\eunpa\anaconda3\envs\sy\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


[001] processing: https://www.instagram.com/reels/DYRs6scPeVC/
[Instagram] Extracting URL: https://www.instagram.com/reels/DYRs6scPeVC/
[Instagram] DYRs6scPeVC: Setting up session
[Instagram] DYRs6scPeVC: Downloading JSON metadata
[info] DYRs6scPeVC: Downloading 1 format(s): 1
Deleting existing file C:\Temp\shot-classification\videos_new_links\new_001_DYRs6scPeVC.mp4
[download] Destination: C:\Temp\shot-classification\videos_new_links\new_001_DYRs6scPeVC.mp4
[download] 100% of   10.44MiB in 00:00:00 at 23.68MiB/s  


overlay saved: C:\Temp\shot-classification\outputs_final_instagram_pipeline\new_001_DYRs6scPeVC\new_001_DYRs6scPeVC_overlay.mp4
성공: 1
실패: 0


,url,shortcode,downloaded_video_path,output_dir,overlay_path,scene_count,duration_sec,threshold,text_scene_count,shot_label_counts
0,https://www.instagram.com/reels/DYRs6scPeVC/,DYRs6scPeVC,C:\Temp\shot-classification\videos_new_links\n...,C:\Temp\shot-classification\outputs_final_inst...,C:\Temp\shot-classification\outputs_final_inst...,25,77.233333,0.137457,24,"{""medium"": 19, ""object"": 5, ""close-up"": 1}"


## 3. 실패 링크 확인

인스타그램 링크가 로그인 없이 접근 불가하거나 삭제/비공개 상태이면 다운로드 단계에서 실패할 수 있습니다.




In [ ]:
failed_df




## 4. 생성된 overlay 영상 경로 확인




In [4]:
if not summary_df.empty:
    display(summary_df[["shortcode", "scene_count", "duration_sec", "text_scene_count", "shot_label_counts", "overlay_path"]])
else:
    print("생성된 overlay 영상이 없습니다.")




,shortcode,scene_count,duration_sec,text_scene_count,shot_label_counts,overlay_path
0,DYRs6scPeVC,25,77.233333,24,"{""medium"": 19, ""object"": 5, ""close-up"": 1}",C:\Temp\shot-classification\outputs_final_inst...
